In [8]:
pip install google-play-scraper pandas nltk stanza seaborn matplotlib scikit-learn Sastrawi

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# **1. Scraping Review Aplikasi TIX ID**

In [9]:
from google_play_scraper import reviews_all, Sort
import pandas as pd

tix_reviews = reviews_all(
    'id.tix.android',
    lang='id',
    country='id',
    sort=Sort.NEWEST
)

df = pd.DataFrame(tix_reviews)
df = df[['content', 'score', 'at']]
df = df.dropna(subset=['content'])
df = df[df['content'].str.strip() != '']

print(f'Total Review: {len(df)}')
df.head()

df.to_excel('tix_id_reviews.xlsx', index=False)

Total Review: 96276


# **2. Label Sentiment dari Rating**

In [10]:
def label_sentiment(score: int) -> str:
  if score >= 4:
    return 'Positive'
  elif score == 3:
    return 'Neutral'
  else:
    return 'Negative'

df['sentiment'] = df['score'].apply(label_sentiment)
df.head()

,content,score,at,sentiment
0,cara riset ngulang nya gimana si soal nya nomo...,4,2026-03-30 23:39:45,Positive
1,Hadirkan fitur Dark Mode dan Navigas bawah sin...,2,2026-03-30 01:21:40,Negative
2,usefull.,5,2026-03-28 13:57:28,Positive
3,jelek ah kenapa uang refund nya ga kembali?,1,2026-03-28 13:49:29,Negative
4,"proses payment gateway qris nya gangguan, udah...",1,2026-03-28 08:22:15,Negative


# **2. Label Sentiment dari Rating**

In [11]:
issue_patterns = {
	'login_issue': r'login|masuk|signin',
	'payment_issue': r'bayar|pembayaran|gagal bayar',
	'error_issue': r'error|bug|crash|force close',
	'performance_issue': r'lambat|lemot|loading lama'
}

for issue, pattern in issue_patterns.items():
	df[issue] = df['content'].str.lower().str.contains(pattern, regex=True)
	
df_neg = df[df['sentiment'] == 'Negative']

issue_summary_neg = {
	issue: df_neg[issue].sum()
	for issue in issue_patterns
}

print(issue_summary_neg)

{'login_issue': np.int64(556), 'payment_issue': np.int64(682), 'error_issue': np.int64(425), 'performance_issue': np.int64(354)}


# **3. Regex Cleaning**

In [12]:
import re

def clean_text(text: str) -> str:
  text = text.lower()
  text = re.sub(r'http\S+', '', text)
  text = re.sub(r'@\w+', '', text)
  text = re.sub(r'[^a-zA-Z\s]', ' ', text)
  text = re.sub(r'(.)\1{2,}', r'\1', text)
  text = re.sub(r'\s+', ' ', text).strip()
  return text

df['clean_content'] = df['content'].apply(clean_text)

df.head()

,content,score,at,sentiment,login_issue,payment_issue,error_issue,performance_issue,clean_content
0,cara riset ngulang nya gimana si soal nya nomo...,4,2026-03-30 23:39:45,Positive,False,False,False,False,cara riset ngulang nya gimana si soal nya nomo...
1,Hadirkan fitur Dark Mode dan Navigas bawah sin...,2,2026-03-30 01:21:40,Negative,False,False,False,False,hadirkan fitur dark mode dan navigas bawah sin...
2,usefull.,5,2026-03-28 13:57:28,Positive,False,False,False,False,usefull
3,jelek ah kenapa uang refund nya ga kembali?,1,2026-03-28 13:49:29,Negative,False,False,False,False,jelek ah kenapa uang refund nya ga kembali
4,"proses payment gateway qris nya gangguan, udah...",1,2026-03-28 08:22:15,Negative,False,False,False,False,proses payment gateway qris nya gangguan udah ...


# **4. Stopword Removal**

In [13]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop = stopwords.words('indonesian')

df['no_stopwords'] = df['clean_content'].apply(
    lambda x: ' '.join([word for word in x.split() if word not in stop])
)

df.head()

[nltk_data] Downloading package stopwords to C:\Users\Ratna
[nltk_data]     Amalia\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,content,score,at,sentiment,login_issue,payment_issue,error_issue,performance_issue,clean_content,no_stopwords
0,cara riset ngulang nya gimana si soal nya nomo...,4,2026-03-30 23:39:45,Positive,False,False,False,False,cara riset ngulang nya gimana si soal nya nomo...,riset ngulang nya gimana si nya nomor telpon n...
1,Hadirkan fitur Dark Mode dan Navigas bawah sin...,2,2026-03-30 01:21:40,Negative,False,False,False,False,hadirkan fitur dark mode dan navigas bawah sin...,hadirkan fitur dark mode navigas sinkronkan wa...
2,usefull.,5,2026-03-28 13:57:28,Positive,False,False,False,False,usefull,usefull
3,jelek ah kenapa uang refund nya ga kembali?,1,2026-03-28 13:49:29,Negative,False,False,False,False,jelek ah kenapa uang refund nya ga kembali,jelek ah uang refund nya ga
4,"proses payment gateway qris nya gangguan, udah...",1,2026-03-28 08:22:15,Negative,False,False,False,False,proses payment gateway qris nya gangguan udah ...,proses payment gateway qris nya gangguan udah ...


# **5. Tokenization**

In [14]:
def tokenize(text: str):
  return text.split()

df['tokens'] = df['no_stopwords'].apply(tokenize)

df.head()

,content,score,at,sentiment,login_issue,payment_issue,error_issue,performance_issue,clean_content,no_stopwords,tokens
0,cara riset ngulang nya gimana si soal nya nomo...,4,2026-03-30 23:39:45,Positive,False,False,False,False,cara riset ngulang nya gimana si soal nya nomo...,riset ngulang nya gimana si nya nomor telpon n...,"[riset, ngulang, nya, gimana, si, nya, nomor, ..."
1,Hadirkan fitur Dark Mode dan Navigas bawah sin...,2,2026-03-30 01:21:40,Negative,False,False,False,False,hadirkan fitur dark mode dan navigas bawah sin...,hadirkan fitur dark mode navigas sinkronkan wa...,"[hadirkan, fitur, dark, mode, navigas, sinkron..."
2,usefull.,5,2026-03-28 13:57:28,Positive,False,False,False,False,usefull,usefull,[usefull]
3,jelek ah kenapa uang refund nya ga kembali?,1,2026-03-28 13:49:29,Negative,False,False,False,False,jelek ah kenapa uang refund nya ga kembali,jelek ah uang refund nya ga,"[jelek, ah, uang, refund, nya, ga]"
4,"proses payment gateway qris nya gangguan, udah...",1,2026-03-28 08:22:15,Negative,False,False,False,False,proses payment gateway qris nya gangguan udah ...,proses payment gateway qris nya gangguan udah ...,"[proses, payment, gateway, qris, nya, gangguan..."


# **6. Stemming**

In [15]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stemming(text: str):
  return stemmer.stem(text)

df['stemmed'] = df['no_stopwords'].apply(stemming)

df.head()

,content,score,at,sentiment,login_issue,payment_issue,error_issue,performance_issue,clean_content,no_stopwords,tokens,stemmed
0,cara riset ngulang nya gimana si soal nya nomo...,4,2026-03-30 23:39:45,Positive,False,False,False,False,cara riset ngulang nya gimana si soal nya nomo...,riset ngulang nya gimana si nya nomor telpon n...,"[riset, ngulang, nya, gimana, si, nya, nomor, ...",riset ngulang nya gimana si nya nomor telpon n...
1,Hadirkan fitur Dark Mode dan Navigas bawah sin...,2,2026-03-30 01:21:40,Negative,False,False,False,False,hadirkan fitur dark mode dan navigas bawah sin...,hadirkan fitur dark mode navigas sinkronkan wa...,"[hadirkan, fitur, dark, mode, navigas, sinkron...",hadir fitur dark mode navigas sinkron warna la...
2,usefull.,5,2026-03-28 13:57:28,Positive,False,False,False,False,usefull,usefull,[usefull],usefull
3,jelek ah kenapa uang refund nya ga kembali?,1,2026-03-28 13:49:29,Negative,False,False,False,False,jelek ah kenapa uang refund nya ga kembali,jelek ah uang refund nya ga,"[jelek, ah, uang, refund, nya, ga]",jelek ah uang refund nya ga
4,"proses payment gateway qris nya gangguan, udah...",1,2026-03-28 08:22:15,Negative,False,False,False,False,proses payment gateway qris nya gangguan udah ...,proses payment gateway qris nya gangguan udah ...,"[proses, payment, gateway, qris, nya, gangguan...",proses payment gateway qris nya ganggu udah da...


# **7. POS Tagging dgn Stanza**

In [16]:
import stanza

stanza.download('id')
nlp = stanza.Pipeline('id')

def pos_tagging(text: str):
  doc = nlp(text)
  result = []
  for sentence in doc.sentences:
    for word in sentence.words:
      result.append((word.text, word.upos))
  return result

print(pos_tagging(df['no_stopwords'].iloc[0]))

c:\Coding\pba\PBA-TIXID-SentimentAnalysis\Week-2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-01 08:43:50 INFO: Downloaded file to C:\Users\Ratna Amalia\AppData\Local\StanfordNLP\stanza\Cache\1.11.0\resources\resources.json
2026-04-01 08:43:50 INFO: Downloading default packages for language: id (Indonesian) ...
2026-04-01 08:43:51 INFO: File exists: C:\Users\Ratna Amalia\AppData\Local\StanfordNLP\stanza\Cache\1.11.0\resources\id\default.zip
2026-04-01 08:43:55 INFO: Finished downloading models and saved to C:\Users\Ratna Amalia\AppData\Local\StanfordNLP\stanza\Cache\1.11.0\resources
2026-04-01 08:43:55 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026

[('riset', 'NOUN'), ('ngulang', 'VERB'), ('nya', 'PRON'), ('gimana', 'ADV'), ('si', 'DET'), ('nya', 'PRON'), ('nomor', 'NOUN'), ('telpon', 'NOUN'), ('nya', 'PRON'), ('ngga', 'X'), ('plis', 'X'), ('pake', 'X'), ('riset', 'NOUN')]


# **8. Hitung Distribusi POS**

In [ ]:
from collections import Counter

df_pos_sample = df.sample(600, random_state=42)

def get_pos_counts(text: str):
  doc = nlp(text)
  pos_list = []
  for sentence in doc.sentences:
    for word in sentence.words:
      pos_list.append(word.upos)
  return Counter(pos_list)

df_pos_sample['pos_counts'] = df['no_stopwords'].apply(get_pos_counts)
df.head()

# **9. Hitung Adjectve**

In [ ]:
def count_adj(text: str) -> int:
  doc = nlp(text)
  count = 0
  for sentence in doc.sentences:
    for word in sentence.words:
      if word.upos == 'ADJ':
        count += 1
  return count

df_pos_sample['adj_count'] = df_pos_sample['no_stopwords'].apply(count_adj)
df.head()

# **10. Visualisasi Rating**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure()
sns.countplot(x='score', data=df)
plt.title('Distribus Rating Aplikasi TIX ID')
plt.show()

# **11. TF-IDF Feature Extraction**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

X = vectorizer.fit_transform(df['stemmed'])
y = df['sentiment']

# **12. Bag of Words (BoW)**

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

df_pos = df[df['sentiment'] == 'Positive']
df_neg = df[df['sentiment'] == 'Negative']

bow_pos = CountVectorizer(max_features=50)
X_pos = bow_pos.fit_transform(df_pos['stemmed'])

df_bow_pos = pd.DataFrame(
	X_pos.toarray(),
	columns=bow_pos.get_feature_names_out()
)

print(df_bow_pos.head())

bow_neg = CountVectorizer(max_features=50)
X_neg = bow_neg.fit_transform(df_neg['stemmed'])

df_bow_neg = pd.DataFrame(
	X_neg.toarray(),
	columns=bow_neg.get_feature_names_out()
)

print(df_bow_neg.head())

df['year'] = pd.to_datetime(df['at']).dt.year

reviews_per_year = df['year'].value_counts().sort_index()

print(reviews_per_year)

plt.figure()
reviews_per_year.plot(kind='bar')
plt.title('Jumlah Review per Tahun')
plt.xlabel('Tahun')
plt.ylabel('Jumlah Review')
plt.show()

pos_word_counts = X_pos.toarray().sum(axis=0)

df_top_pos = pd.DataFrame({
	'word': bow_pos.get_feature_names_out(),
	'count': pos_word_counts
}).sort_values(by='count', ascending=False)

print(df_top_pos.head(10))

neg_word_counts = X_neg.toarray().sum(axis=0)

df_top_neg = pd.DataFrame({
	'word': bow_neg.get_feature_names_out(),
	'count': neg_word_counts
}).sort_values(by='count', ascending=False)

print(df_top_neg.head(10))

# **13. N-Gram Feature**

In [ ]:
ngram_vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    max_features=5000
)

X_ngram = ngram_vectorizer.fit_transform(df['stemmed'])

print("N-Gram Shape:", X_ngram.shape)

# **14. Train, Test, Split**

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# **15. Model 1 - Naive Bayes**

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

nb = MultinomialNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

# **16. Model 2 - Logistic Regression**

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

print ("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

# **17. Model 3 - SVM**

In [ ]:
from sklearn.svm import LinearSVC
svm = LinearSVC()
svm.fit(X_train, y_train)

y_pred_svm = svm.predict(X_test)

print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

# **18. Perbandingan Accuracy Model**

In [ ]:
results = {
    "Naive Bayes": accuracy_score(y_test, y_pred_nb),
    "Logistic Regression": accuracy_score(y_test, y_pred_lr),
    "SVM": accuracy_score(y_test, y_pred_svm),
}

print(results)

# **19. Visualisasi Perbandingan Model**

In [ ]:
plt.figure()
plt.bar(results.keys(), results.values())
plt.title("Perbandingan Accuracy Model Sentiment Analysis")
plt.ylabel("Accuracy")
plt.show()